# 01 Data Cleaning
Tena Alshaikhli · AAI-501 Final Team Project (Team 3)

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/usd-tenaalshaikhli/AAI501-Team3-Ecommerce-Churn/main/data/E%20Commerce%20Dataset.xlsx"
df = pd.read_excel(url, sheet_name="E Comm")   # the data is on the "E Comm" sheet

print(df.shape)
df.head()

(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [2]:
# Count missing values in each column
missing = df.isnull().sum()
missing[missing > 0]

Tenure                         264
WarehouseToHome                251
HourSpendOnApp                 255
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
dtype: int64

In [3]:
# Fix labels that mean the same thing
fixes = {
    "PreferredLoginDevice": {"Phone": "Mobile Phone"},
    "PreferredPaymentMode": {"CC": "Credit Card", "COD": "Cash on Delivery"},
    "PreferedOrderCat": {"Mobile": "Mobile Phone"},
}
df = df.replace(fixes)

for col in fixes:
    print(col, df[col].value_counts().to_dict())

PreferredLoginDevice {'Mobile Phone': 3996, 'Computer': 1634}
PreferredPaymentMode {'Debit Card': 2314, 'Credit Card': 1774, 'E wallet': 614, 'Cash on Delivery': 514, 'UPI': 414}
PreferedOrderCat {'Mobile Phone': 2080, 'Laptop & Accessory': 2050, 'Fashion': 826, 'Grocery': 410, 'Others': 264}


In [4]:
# Remove duplicates BEFORE filling missing values, so filled values can't create new duplicates
cols_except_id = [c for c in df.columns if c != "CustomerID"]

print("Duplicate rows found:", df.duplicated(subset=cols_except_id).sum())
df = df.drop_duplicates(subset=cols_except_id, keep="first").reset_index(drop=True)

print("Rows after removing duplicates:", df.shape[0])
print("Churn rate now: {:.1f}%".format(df["Churn"].mean() * 100))

Duplicate rows found: 557
Rows after removing duplicates: 5073
Churn rate now: 16.6%


In [5]:
# Fill missing values with each column's median
missing_cols = df.columns[df.isnull().any()].tolist()
medians = df[missing_cols].median()
print("Medians used:", medians.to_dict())

df[missing_cols] = df[missing_cols].fillna(medians)
print("Missing values left:", df.isnull().sum().sum())

Medians used: {'Tenure': 9.0, 'WarehouseToHome': 13.0, 'HourSpendOnApp': 3.0, 'OrderAmountHikeFromlastYear': 15.0, 'CouponUsed': 1.0, 'OrderCount': 2.0, 'DaySinceLastOrder': 3.0}
Missing values left: 0


In [6]:
# Save the cleaned dataset
df.to_csv("ecommerce_churn_clean.csv", index=False)
print("Saved ecommerce_churn_clean.csv:", df.shape)

try:
    from google.colab import files   # downloads the file when run in Google Colab
    files.download("ecommerce_churn_clean.csv")
except ImportError:
    pass

Saved ecommerce_churn_clean.csv: (5073, 20)
